# Récord de puntos por jugador

Sistemas de Big Data 5074 · UT1.2 · Diapositivas 17–22.

Versión para completar. Puedes trabajar en este cuaderno en lugar de `alumno.py`. Sustituye los `NotImplementedError` por tu código. Las celdas con tareas pendientes se detendrán hasta que las completes.

Solo biblioteca estándar de Python 3.10 o posterior. Los datos están incluidos. No requiere instalaciones ni archivos externos. Puedes subir este `.ipynb` a Colab. Los ejemplos MapReduce se simulan localmente y en serie, sin ejecutar Hadoop.


# Ejemplo 3. Récord por jugador

Vas a completar el ejemplo de las diapositivas 17–22 de `Evaluacion_UT1_2.pptx`, de UT1.2 de SBD (5074). El objetivo es obtener la puntuación máxima de **cada jugador** entre varios archivos de partidas.

Abre `alumno.py` y completa sus cinco TODO. Los bloques `# %%` permiten avanzar por etapas en un editor que reconozca celdas. Con Python 3.10 o posterior también puedes ejecutar el archivo completo:

```console
python3 alumno.py
```

Al principio se detendrá con `NotImplementedError`. Sustituye cada instrucción de ese tipo por el código que pide su comentario. Los imports, los datos y la visualización final ya están preparados. Solo necesitas la biblioteca estándar.

## Datos de la práctica

Cada línea contiene `fecha jugador puntos`. Las tres primeras líneas de mayo reproducen el texto visible de la diapositiva 19. La cuarta línea de mayo y las tres de junio amplían los datos para obtener las listas de la diapositiva 22. **Las fechas de esos cuatro registros añadidos son ficticias.**

El diccionario `FICHEROS` contiene todos los datos. Los archivos de `datos/` son una copia para consultarlos. El programa utiliza el diccionario y puede ejecutarse desde cualquier carpeta sin leer archivos externos. Ambos meses están en UTF-8, sin BOM y con saltos de línea LF.

## Bloque de lectura: TODO 1

Completa `leer_registros(texto)`. Devuelve una lista de pares `(offset, linea)`. Usa `BytesIO`, consulta `tell()` **antes** de cada `readline()` y decodifica cada línea con UTF-8. Retira el terminador de línea al construir el texto del registro.

El offset indica cuántos bytes hay antes del comienzo de la línea dentro de ese archivo. Calcula los offsets de mayo a partir de la lectura binaria. La `Í` ocupa dos bytes en UTF-8. Si el texto usa CRLF, cuenta los dos bytes del salto. La última línea también se procesa cuando no termina en LF.

Cada archivo empieza en el byte 0. Este valor no representa la fecha ni la posición en una lista, y tampoco identifica un registro en toda la colección. La diapositiva 21 imprime `0, 23, 48`, pero esos números no se reproducen con el texto visible normalizado a UTF-8 y LF que usamos aquí.

## Bloque Map: TODO 2

Completa `mapa_registro(offset, linea)`. Extrae la puntuación desde la derecha con `rsplit(" ", 1)` y separa la fecha del jugador con `split(maxsplit=1)`. Así el nombre puede contener espacios. Devuelve, por ejemplo, `("MARÍA", 18)`.

La fecha debe ser válida y los puntos deben ser enteros. Si el registro está vacío o tiene un formato incorrecto, señala un `ValueError` con la línea y su offset. Las puntuaciones negativas sí se aceptan.

## Bloque Shuffle: TODO 3

Completa `agrupar(pares)`. Usa el jugador como clave de un diccionario y añade sus puntuaciones a una lista. Construye los grupos con los siete registros y comprueba que cada puntuación llega al jugador correspondiente.

## Bloque Reduce: TODO 4

Completa `reducir(jugador, puntos)`. Recorre los puntos para calcular el máximo y devuelve `(jugador, maximo)`. Empieza por la primera puntuación para que también funcione si todas son negativas. Una lista vacía debe provocar `ValueError`.

En un empate devolvemos la puntuación máxima una sola vez. El resultado no incluye fechas ni partidas ganadoras.

## Bloque de ejecución: TODO 5

Completa la coordinación dentro de `ejecutar_mapreduce(ficheros)`. Recorre todos los archivos, lee sus registros y aplica Map. Mantén una emisión por registro. Cuando haya un error, añade el nombre del archivo y vuelve a lanzarlo.

Después conecta Shuffle y Reduce: agrupa las emisiones, recorre los grupos y guarda el máximo de cada jugador en `resultados`. Para seguir el proceso en clase, puedes mostrar las emisiones y los grupos dentro de esta función.

Cuando termines, ejecuta el programa. El bloque final llama a tus funciones para mostrar la lectura y comparar los récords calculados con los tres registros originales y con los siete de la práctica. Revisa manualmente qué partidas explican los resultados.

Cada etiqueta de jugador se considera una misma identidad en todos los archivos. `MARÍA` y `María` formarían claves diferentes. Estamos calculando un récord por jugador. Esta simulación trabaja en serie en tu ordenador y guarda los grupos en memoria. Hadoop distribuye el trabajo y gestiona aspectos que aquí no implementamos.



## Código 1 Datos de la práctica


In [48]:
"""Ejemplo 3: completa las cinco partes indicadas con TODO.

Python 3.10 o posterior. No necesitas instalar paquetes ni abrir los .txt.
Las tres primeras líneas de mayo proceden de la diapositiva 19.
La cuarta línea de mayo y las fechas de junio son inventadas para practicar.
Sus puntuaciones completan las listas de la diapositiva 22.
"""

from datetime import datetime
from io import BytesIO


FICHEROS = {
    "Mayo_2020.txt": (
        "01-05-2020 MARÍA 18\n"
        "03-05-2020 JUAN 23\n"
        "07-05-2020 MARÍA 45\n"
        "12-05-2020 JUAN 5\n"
    ),
    "Junio_2020.txt": (
        "02-06-2020 MARÍA 30\n"
        "09-06-2020 MARÍA 60\n"
        "16-06-2020 JUAN 45\n"
    ),
}


## Código 2 Lectura de líneas y offsets en bytes


In [49]:
from os import linesep
def leer_registros(texto: str) -> list[tuple[int, str]]:
    """Devuelve (offset en bytes, línea sin terminador) por cada línea."""
    flujo = BytesIO(texto.encode("utf-8"))
    registros = []
    # TODO 1: antes de leer cada línea, guarda flujo.tell().
    # Usa flujo.readline() y termina al recibir b"".
    # Decodifica UTF-8, quita solo los terminadores \r y \n y añade el par.
    # Conserva las líneas vacías para que Map pueda señalar el error.
    # raise NotImplementedError("TODO 1: leer líneas con su offset en bytes")

    while True:
      offset = flujo.tell()
      linea = flujo.readline()
      if linea == b"":
        break

      registro = linea.decode("utf-8").rstrip('\r\n')
      registros.append((offset, registro))

    flujo.close()

    return registros


## Código 3 Map: de una línea a un par jugador y puntos


In [50]:
def mapa_registro(offset: int, linea: str) -> tuple[str, int]:
    """Devuelve (jugador, puntos). Un registro inválido provoca ValueError."""
    # TODO 2: separa los puntos con linea.strip().rsplit(" ", 1).
    # Separa la fecha del jugador con split(maxsplit=1).
    # Comprueba la fecha con datetime.strptime(fecha, "%d-%m-%Y").
    # Convierte los puntos a int. Acepta nombres compuestos y puntos negativos.
    # Si el formato falla, lanza ValueError que incluya el offset y la línea.
    # raise NotImplementedError("TODO 2: transformar un registro")
    fecha_jugador, puntuacion_in = linea.strip().rsplit(" ", 1)
    fecha, jugador = fecha_jugador.split(maxsplit=1)

    if not datetime.strptime(fecha, "%d-%m-%Y"):
      raise ValueError(f"La fecha no es correcta en {offset}: {linea}")

    try:
      puntuacion_out = int(puntuacion_in)
    except ValueError:
      raise ValueError("fLa puntuación no es un número entero en {offset}: {linea}")

    return jugador, puntuacion_out




## Código 4 Shuffle: reunir los puntos de cada jugador


In [51]:
def agrupar(pares: list[tuple[str, int]]) -> dict[str, list[int]]:
    """Devuelve un diccionario de jugador a lista de puntuaciones."""
    grupos = {}
    # TODO 3: recorre los pares. Crea la lista si el jugador aún no aparece
    # y añade sus puntos a la lista correspondiente.
    # raise NotImplementedError("TODO 3: agrupar las emisiones por jugador")

    for par in pares:
      jugador, puntuacion = par
      if jugador not in grupos:
        grupos[jugador] = []
        print(f"  Jugador {jugador} añadido")
      grupos[jugador].append(puntuacion)
      print(f"  {jugador}, puntuación {puntuacion} añadida")

    return grupos


## Código 5 Reduce: máximo de los puntos de un jugador


In [52]:
def reducir(jugador: str, puntos: list[int]) -> tuple[str, int]:
    """Devuelve (jugador, máximo). Una lista vacía provoca ValueError."""
    # TODO 4: comprueba que haya puntos. Inicia el máximo con el primer valor
    # y recorre los demás para actualizarlo cuando corresponda.
    # Si dos puntuaciones empatan, el resultado sigue siendo ese mismo número.
    # raise NotImplementedError("TODO 4: calcular el máximo del jugador")

    if len(puntos) < 1:
      raise ValueError("Puntos debe ser una lista no vacia")

    maximo = puntos[0]

    print(f"    {maximo}")
    for puntuacion in puntos[1:]:
      if maximo < puntuacion:
        maximo = puntuacion
        print(f"    {maximo}")

    return jugador, maximo


## Código 6 Ejecución de MapReduce para una colección de archivos


In [53]:
def ejecutar_mapreduce(ficheros: dict[str, str]) -> dict[str, int]:
    """Conecta las etapas de lectura, Map, Shuffle y Reduce."""
    emisiones = []
    resultados = {}
    # TODO 5: recorre ficheros.items() y los registros de leer_registros(texto).
    # Llama a mapa_registro(offset, linea) y añade cada par a emisiones.
    # Ante ValueError, vuelve a lanzar el error añadiendo el nombre del archivo.
    # El offset empieza en 0 en cada archivo. Evita usarlo como clave global.
    # Después llama a agrupar. Recorre cada grupo y llama a reducir.
    # Guarda cada jugador y su máximo en resultados.
    # Puedes mostrar emisiones y grupos aquí para seguir el proceso en clase.
    # raise NotImplementedError("TODO 5: coordinar las etapas de MapReduce")

    print("Leer ficheros")
    for fichero, texto in FICHEROS.items():
      print(f"  Leyendo fichero {fichero}")
      for registro in leer_registros(texto):
        offset, linea = registro
        print(f"  {offset}: {linea}")
        try:
          emision = mapa_registro(offset, linea)
        except Exception as e:
          print(f"Error en el fichero {fichero}:{e}")
        else:
          emisiones.append(emision)

    print("\nAgrupar emisiones")
    jugador_puntuacion = agrupar(emisiones)

    print(f"\nReducir puntuaciones")
    for jugador, puntuaciones in jugador_puntuacion.items():
      print(f"  Evolución de puntuaciones máximas de {jugador}")
      try:
        puntuacion_maxima = reducir(jugador, puntuaciones)
      except ValueError:
        print(f"Error, lista de puntuaciones del jugador {jugador} está vacia")
      else:
        resultados[jugador] = puntuacion_maxima

    print("\nResultados")
    for jugador, puntuacion in resultados.items():
      print(f"  {jugador}: {puntuacion}")

    return resultados


## Código 7 Observación del proceso y de los resultados


In [54]:
if __name__ == "__main__":
    print("Offsets y líneas de Mayo_2020.txt (UTF-8, LF):")
    print(leer_registros(FICHEROS["Mayo_2020.txt"]))

    tres_lineas = FICHEROS["Mayo_2020.txt"].splitlines(keepends=True)[:3]
    originales = {"Mayo_2020.txt": "".join(tres_lineas)}
    print("\nRécords con los tres registros originales:")
    print(ejecutar_mapreduce(originales))
    print("\nRécords con los siete registros de la práctica:")
    print(ejecutar_mapreduce(FICHEROS))


Offsets y líneas de Mayo_2020.txt (UTF-8, LF):
[(0, '01-05-2020 MARÍA 18'), (21, '03-05-2020 JUAN 23'), (40, '07-05-2020 MARÍA 45'), (61, '12-05-2020 JUAN 5')]

Récords con los tres registros originales:
Leer ficheros
  Leyendo fichero Mayo_2020.txt
  0: 01-05-2020 MARÍA 18
  21: 03-05-2020 JUAN 23
  40: 07-05-2020 MARÍA 45
  61: 12-05-2020 JUAN 5
  Leyendo fichero Junio_2020.txt
  0: 02-06-2020 MARÍA 30
  21: 09-06-2020 MARÍA 60
  42: 16-06-2020 JUAN 45

Agrupar emisiones
  Jugador MARÍA añadido
  MARÍA, puntuación 18 añadida
  Jugador JUAN añadido
  JUAN, puntuación 23 añadida
  MARÍA, puntuación 45 añadida
  JUAN, puntuación 5 añadida
  MARÍA, puntuación 30 añadida
  MARÍA, puntuación 60 añadida
  JUAN, puntuación 45 añadida

Reducir puntuaciones
  Evolución de puntuaciones máximas de MARÍA
    18
    45
    60
  Evolución de puntuaciones máximas de JUAN
    23
    45

Resultados
  MARÍA: ('MARÍA', 60)
  JUAN: ('JUAN', 45)
{'MARÍA': ('MARÍA', 60), 'JUAN': ('JUAN', 45)}

Récords con 

## Explicación de tus resultados

Anota aquí tus conclusiones y las respuestas que pide el guion.
